In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

In [3]:
def length_function(text):
    return len(text)

def _multiple_length_function(text1, text2):
    return len(text1) * len(text2)

def multiple_length_function(
        _dict,
):
    return _multiple_length_function(_dict["text1"], _dict["text2"])

prompt = ChatPromptTemplate.from_template("what is {a} + {b}?")
model = ChatOpenAI()
chain = (
    {
        "a": itemgetter("input_1") | RunnableLambda(length_function),
        "b": {"text1": itemgetter("input_1"), "text2": itemgetter("input_2")}
        | RunnableLambda(multiple_length_function),
    }
    | prompt
    | model
    | StrOutputParser()
)

In [7]:
chain.invoke({"input_1": "bar", "input_2": "gah"})

'3 + 9 = 12'

In [8]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableConfig
import json

In [10]:
def parse_or_fix(text: str, config: RunnableConfig):
    fixing_chain = (
        ChatPromptTemplate.from_template(
            "Fix the following text:\n\ntext\n{input}\n\nError: {error}"
            "Don't narrate, just respond with the fixed data."
        )
        | ChatOpenAI()
        | StrOutputParser()
    )
    for _ in range(3):
        try:
            return json.loads(text)
        except Exception as e:
            text = fixing_chain.invoke({"input": text, "error": e}, config)
            print(f"config: {config}")

    return "Failed to parse"

In [11]:
from langchain_classic.callbacks import get_openai_callback

with get_openai_callback() as cb:
    output = RunnableLambda(parse_or_fix).invoke(
        input="{foo::bar}",
        config={"tags": ["by-tag"], "callbacks": [cb]},
    )

    print(f"\n\n수정한 결과:\n{output}")

config: {'tags': ['by-tag'], 'metadata': {}, 'callbacks': <langchain_core.callbacks.manager.CallbackManager object at 0x00000180608C7170>, 'recursion_limit': 25, 'configurable': {}}


수정한 결과:
{'foo': 'bar'}


In [12]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

In [13]:
prompt = PromptTemplate.from_template(
    """주어진 사용자 질문을 `수학`, `과학`, 또는 `기타` 중 하나로 분류하세요. 한 단어 이상
    으로 응답하지 마세요.
    
    <question>
    {question}
    </question>
    
    Classification:"""
)

chain = (
    prompt
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)

In [14]:
chain.invoke({"question": "2+2는 무엇인가요?"})

'수학'

In [15]:
chain.invoke({"question": "작용 반작용의 법칙은 무엇인가요?"})

'과학'

In [16]:
chain.invoke({"question": "Google은 어떤 회사인가요?"})

'기타'

In [17]:
math_chain = (
    PromptTemplate.from_template(
        """You are an expert in math. \
            Always answer questions starting with "깨봉 선생님께서 말씀하시기를..". \
            Respond to the following question:
            
            Question: {question}
            Answer:"""
    )
    | ChatOpenAI(model="gpt-4o-mini")
)

science_chain = (
    PromptTemplate.from_template(
        """You are an expert in science. \
            Always answer questions starting with "아이작 뉴턴 선생님께서 말씀하시기를..". \
            Respond to the following question:
            
            Question: {question}
            Answer:"""
    )
    | ChatOpenAI(model="gpt-4o-mini")
)

general_chain = (
    PromptTemplate.from_template(
        """Respond to the following question concisely:
        
        Question: {question}
        Answer:"""
    )
    | ChatOpenAI(model="gpt-4o-mini")
)

In [18]:
def route(info):
    if "수학" in info["topic"].lower():
        return math_chain
    elif "과학" in info["topic"].lower():
        return science_chain
    else:
        return general_chain

In [19]:
from operator import itemgetter
from langchain_core.runnables import RunnableLambda

full_chain = (
    {"topic": chain, "question": itemgetter("question")}
    | RunnableLambda(
        route
    )
    | StrOutputParser()
)

In [20]:
full_chain.invoke({"question": "미적분의 개념에 대해 말씀해 주세요."})

'깨봉 선생님께서 말씀하시기를, 미적분은 두 가지 주요 개념인 미분과 적분으로 구성되어 있습니다. 미분은 함수의 변화율을 측정하는 도구로, 주어진 함수의 기울기를 구하는 데 사용됩니다. 즉, 어떤 함수가 특정 점에서 얼마나 빠르게 변화하는지를 나타내죠.\n\n반면에, 적분은 함수의 넓이 또는 총합을 계산하는 방법입니다. 이는 주어진 구간에서 함수의 값을 모두 합산하여 면적을 구하는 과정으로, 미분의 역과정이라고도 할 수 있습니다. 미적분학은 이러한 두 개념 간의 깊은 관계를 탐구하며, 물리학, 공학, 경제학 등 다양한 분야에서 응용됩니다. \n\n따라서 미적분은 수학의 근본적인 도구로서, 변화와 양을 이해하고 분석하는 데 필수적인 역할을 합니다.'

In [22]:
full_chain.invoke({"question": "중력은 어떻게 작용하나요?"})

'아이작 뉴턴 선생님께서 말씀하시기를, 중력은 모든 물질 사이에 작용하는 힘으로, 두 물체의 질량과 그들 사이의 거리의 제곱에 반비례하여 작용한다고 하셨습니다. 즉, 질량이 클수록 중력의 힘이 강해지고, 두 물체가 멀어질수록 그 힘은 약해집니다. 이를 통해 우리는 물체가 떨어지는 이유와 천체들이 궤도를 유지하는 원리를 이해할 수 있습니다. 중력은 우주에서 물체들이 서로 끌어당기는 기본적인 힘으로, 만물의 움직임을 지배하는 법칙 중 하나입니다.'

In [24]:
full_chain.invoke({"question": "RAG(Retrieval Augmented Generation)은 무엇인가요?"})

'RAG(검색 확장 생성)는 자연어 처리 모델로, 정보 검색과 텍스트 생성을 결합한 방법입니다. 이 모델은 외부 데이터베이스에서 관련 정보를 검색한 후, 이를 바탕으로 더 풍부하고 정확한 응답을 생성합니다. RAG는 질문에 대해 응답할 때, 정답이 될 가능성이 높은 정보를 찾아 활용하여 생성 품질을 향상시킵니다.'

In [25]:
from operator import itemgetter
from langchain_core.runnables import RunnableBranch

branch = RunnableBranch(
    (lambda x: "수학" in x["topic"].lower(), math_chain),
    (lambda x: "과학" in x["topic"].lower(), science_chain),
    general_chain,
)

full_chain = (
    {"topic": chain, "question": itemgetter("question")} | branch | StrOutputParser()
)

In [26]:
full_chain.invoke({"question": "미적분의 개념에 대해 말씀해 주세요."})

'깨봉 선생님께서 말씀하시기를, 미적분은 수학의 한 분야로, 함수의 변화율과 면적을 다루는 학문입니다. 미분은 함수의 접선 기울기를 통해 그 함수의 변화율을 이해하고, 적분은 함수 아래의 면적을 구하는 방법입니다. 미적분은 물리학, 경제학, 공학 등 다양한 분야에서 활용되며, 변화하는 현상을 수학적으로 분석하는 데 필수적인 도구입니다. 예를 들어, 미적분을 통해 이동 거리, 속도, 가속도 등의 관계를 파악할 수 있습니다.'

In [27]:
full_chain.invoke({"question": "중력 가속도는 어떻게 계산하나요?"})

'아이작 뉴턴 선생님께서 말씀하시기를, 중력 가속도는 지구 표면에서 물체가 받는 중력의 세기와 관련이 있습니다. 지구의 중력 가속도 (g)는 대략 9.81 m/s²로 알려져 있으며, 이를 계산하기 위해 다음의 식을 사용합니다:\n\ng = G * (M/r²)\n\n여기서 G는 중력 상수 (약 6.674 × 10⁻¹¹ N(m/kg)²), M은 지구의 질량 (약 5.97 × 10²⁴ kg), r은 지구의 반지름 (약 6.371 × 10⁶ m)입니다. 이 공식을 통해 지구의 중력 가속도를 계산할 수 있습니다.'

In [28]:
full_chain.invoke({"question": "RAG(Retrieval Augmented Generation)은 무엇인가요?"})

'RAG(Recovery Augmented Generation)은 정보 검색과 자연어 생성을 결합한 모델로, 주어진 질문에 대해 관련 정보를 검색한 후 이를 바탕으로 답변을 생성하는 방식입니다. 이를 통해 보다 정확하고 풍부한 답변을 제공할 수 있습니다.'